# 03. Финальная мультимодальная модель

**Цель.** Зафиксировать финальную модель как единую fusion-систему: log branch + KPI branch + modality masks + fused heads. Ноутбук является витриной архитектуры, метрик и экспорта; тяжелая реализация обучения находится в `dashboard.train_multimodal_fusion`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

from dashboard.final_data_pipeline import PROJECT_ROOT, artifact_paths
from dashboard.export_dashboard_bundle import export_bundle

ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
PATHS = artifact_paths(ARTIFACT_DIR)
RUN_ID = "final_multimodal_strict_intensity_gap10"
MODEL_RUN_DIR = PATHS.model_dir / RUN_ID
RUN_TRAIN = os.environ.get("RUN_FINAL_TRAIN", "0") == "1"

train_command = [
    sys.executable,
    "-m", "dashboard.train_multimodal_fusion",
    "--contract", str(PATHS.contract_path),
    "--artifact-dir", str(ARTIFACT_DIR),
    "--run-id", RUN_ID,
    "--seeds", "42,43,44",
]
print("Команда обучения:")
print(" ".join(map(str, train_command)))
print("RUN_FINAL_TRAIN=", RUN_TRAIN)


## 1. Финальная постановка

Модель получает не только текст логов и KPI-признаки, но и маски доступности модальностей. Это важно: отсутствие лога или KPI — отдельный диагностический факт, а не ноль в признаках. Основной score для дашборда строится по intensity-head, потому что полезная тревога должна отражать нарастание риска до события, а не только бинарный факт окна.


In [ ]:
architecture = pd.DataFrame([
    {"Блок": "Log branch", "Роль": "кодирует top log events до текущего окна"},
    {"Блок": "KPI branch", "Роль": "кодирует последовательность KPI за предыдущие окна"},
    {"Блок": "Modality masks", "Роль": "сообщает модели, какие модальности доступны"},
    {"Блок": "Fusion block", "Роль": "объединяет log/KPI представления в один hidden state"},
    {"Блок": "Binary heads", "Роль": "строгие warning/risk цели"},
    {"Блок": "Intensity heads", "Роль": "плавное нарастание риска по lead time"},
])
display(architecture)


**Локальный вывод.** Финальная модель остается строго мультимодальной: одно fused-решение строится из двух ветвей и масок доступности. Log-only и KPI-only используются только как baseline/диагностика, а не как продуктовая система.


## 2. Обучение

По умолчанию notebook не запускает тяжелое обучение при каждом открытии. Для полного воспроизводимого train-run нужно выполнить notebook или команду ниже с переменной окружения `RUN_FINAL_TRAIN=1`. На машине с CUDA можно добавить `--device cuda --text-backend embeddinggemma`.


In [ ]:
if RUN_TRAIN:
    subprocess.run(train_command, check=True)
elif MODEL_RUN_DIR.exists():
    print(f"Найден готовый train-run: {MODEL_RUN_DIR.relative_to(PROJECT_ROOT)}")
else:
    print("Train-run пока не найден. Запустите команду обучения выше или установите RUN_FINAL_TRAIN=1.")


**Локальный вывод.** Обучение вынесено в единый Python-пайплайн. Это делает дашборд заменяемым: если завтра модель улучшается, preprocessing, alert-layer и replay-интерфейс остаются теми же.


## 3. Метрики и сравнение с baseline

После обучения читаем метрики из `dashboard/artifacts/model/<run_id>/`. Главное сравнение: `warning_intensity_score` и `state_score` против baseline на том же strict/intensity contract.


In [ ]:
metrics_path = MODEL_RUN_DIR / "ensemble_metrics.csv"
baseline_path = MODEL_RUN_DIR / "baseline_comparison.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    focus = metrics.loc[
        metrics["score"].isin(["warning_intensity_score", "risk_intensity_score", "state_score"]),
        ["target", "model", "score", "average_precision", "roc_auc", "precision", "recall", "f1"],
    ]
    display(focus)
else:
    print("Метрики модели пока не найдены.")

if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    display(baseline.sort_values(["target", "average_precision"], ascending=[True, False]))
else:
    print("Baseline comparison train-run пока не найден.")


**Локальный вывод.** Primary-оценка модели — ranking/alert usefulness, а не обещание идеального бинарного оракула. Risk-channel остается диагностическим: если он слабее warning-intensity, дашборд должен показывать это честно.


## 4. Экспорт для дашборда

Дашборд не обучает модель и не читает старые исследовательские таблицы. Он использует только компактный bundle: replay-окна, score, alert-state, события run, baseline comparison и model report.


In [ ]:
if (MODEL_RUN_DIR / "ensemble_predictions.csv").exists() and PATHS.contract_path.exists():
    report = export_bundle(model_run_dir=MODEL_RUN_DIR, contract_path=PATHS.contract_path, out_dir=PATHS.bundle_dir)
    print(json.dumps({
        "bundle_dir": str(PATHS.bundle_dir.relative_to(PROJECT_ROOT)),
        "model_name": report.get("model_name"),
        "score_source": report.get("score_source"),
    }, ensure_ascii=False, indent=2))
else:
    print("Bundle не экспортирован: сначала нужен train-run модели и contract из 02.")


**Итоговый handoff.** Для запуска интерфейса нужны только файлы из `dashboard/artifacts/bundle/`. Команда: `streamlit run dashboard/mm_dashboard.py`. Если bundle отсутствует, дашборд показывает команды подготовки и не включает proxy-scoring.
